# Smallprint: end-to-end on Colab or Kaggle

Runs the full Track 1 pipeline: **CuratorKIT** data, **AlignTune** QLoRA SFT and DPO, then **AuditKIT** before/after evaluation.

1. Runtime -> change runtime type -> GPU (a T4 works; an L4 or A100 is faster).
2. On Hugging Face, accept the Tiny Aya licence at https://huggingface.co/CohereLabs/tiny-aya-water.
3. Add these Colab secrets: `HF_TOKEN` (write access) and `COHERE_API_KEY` (teacher and judge). Without a Cohere key, set `NO_TEACHER = True`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone https://github.com/mukeshkbj/smallprint.git
%cd smallprint
!pip -q install -e ".[data,train,eval]"

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    os.environ["COHERE_API_KEY"] = userdata.get("COHERE_API_KEY") or ""
except Exception:
    pass  # Kaggle: use Add-ons -> Secrets, or export them yourself

HF_USER = "your-hf-username"          # must match the username on your hackathon registration
os.environ["SMALLPRINT_HUB__USER"] = HF_USER
NO_TEACHER = not os.environ.get("COHERE_API_KEY")
SMOKE_TEST = True                     # True = tiny run to check everything works; False = the real run

from huggingface_hub import login
login(os.environ["HF_TOKEN"])

## 1. Data (CuratorKIT)

In [ ]:
flags = ("--no-teacher " if NO_TEACHER else "") + ("--limit 300" if SMOKE_TEST else "--push")
!python scripts/01_build_data.py {flags}
!cat data/build_report.json

## 2. Before: zero-shot Tiny Aya Water
Scoring the base model first means you have a baseline even if training runs out of time.

In [ ]:
n = "--max-samples 200" if SMOKE_TEST else ""
!python scripts/05_evaluate.py --skip-final {n}

## 3. SFT (AlignTune, QLoRA)

In [ ]:
steps = "--max-steps 30" if SMOKE_TEST else "--push"
!python scripts/02_train_sft.py {steps}

## 4. DPO pairs (on-policy mistakes plus targeted negatives) and DPO training

In [ ]:
!python scripts/03_build_dpo.py
steps = "--max-steps 20" if SMOKE_TEST else "--push"
!python scripts/04_train_dpo.py {steps}

## 5. After: AuditKIT before/after

In [ ]:
n = "--max-samples 200" if SMOKE_TEST else ""
!python scripts/05_evaluate.py --with-sft {n}
from IPython.display import Markdown, display
display(Markdown(open("results/summary.md").read()))

## 6. Save results to GitHub
Commit `results/*.json`, `results/summary.md` and the AuditKIT comparison files so judges can see them in the repo.